**TransForm Payments Data**

In [0]:
%python
dfPayments=spark.table('gizmobox_gr.bronze.py_payments')
display(dfPayments)

**Extract Date and Time From PaymentOn**

In [0]:
%python
from pyspark.sql.functions import date_format

df_data=dfPayments.withColumns(
                                         "paymentId",
                                         "orderId",
                                         date_format("paymetOn","yyyy-MM-dd").cast("date").alias("payment_date"),
                                         date_format("paymetOn","HH:mm:ss").cast("timestamp").alias("payment_timestamp"),
                                         "paymentStatus",
                                         "paymentMethod")
display(df_data)
# df_payment_datetime_extract=dfPayments.select(
#                                          "paymentId",
#                                          "orderId",
#                                          date_format("paymetOn","yyyy-MM-dd").cast("date").alias("payment_date"),
#                                          date_format("paymetOn","HH:mm:ss").cast("timestamp").alias("payment_timestamp"),
#                                          "paymentStatus",
#                                          "paymentMethod")
# display(df_payment_datetime_extract)

**Map Payment Status to Descriptive Values**

In [0]:
%python
from pyspark.sql import functions as f

df_payment_map_paymentstatus=df_payment_datetime_extract.select(
                                         "paymentId",
                                         "orderId",
                                         "payment_date",
                                         "payment_timestamp",
                                         "paymentMethod",
                                         "paymentStatus",
                                         f.when(df_payment_datetime_extract.paymentStatus==1,"Success")
                                         .when(df_payment_datetime_extract.paymentStatus==2,"Pending")
                                         .when(df_payment_datetime_extract.paymentStatus==3,"Cancelled")
                                         .when(df_payment_datetime_extract.paymentStatus==4,"Failed")
                                         .alias("payment_status")
                                         )
display(df_payment_map_paymentstatus)

**Write Transformed Data To Silver Schema**

In [0]:
%python
df_payment_map_paymentstatus.writeTo("gizmobox_gr.silver.py_payments").createOrReplace()
display(spark.table("gizmobox_gr.silver.py_payments"))